In [5]:
# ── Cell 1: Setup and Imports ───────────────────────────────────────────
import json
import os
import sys
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp

# Add the parent directory to the path so it can find the root folders
sys.path.append('..') 

# Import your centralized modules
from configs.config import Config
from configs.metrics import evaluate
from data.dataset import BraTSDataset
from models.bu_net_multitask import BUNetMultiTask

print(f"✓ Using device: {Config.DEVICE}")
print(f"✓ Data path: {Config.TRAIN_DATASET_PATH}")

✓ Using device: mps
✓ Data path: /Users/yuzheli/.cache/kagglehub/datasets/awsaf49/brats20-dataset-training-validation/versions/1/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData


In [6]:
# ── Cell 2: Data Splitting & Loaders ─────────────────────────────────────
# Fix the BraTS20_Training_355 file naming issue
old_name = os.path.join(Config.TRAIN_DATASET_PATH, "BraTS20_Training_355/W39_1998.09.19_Segm.nii")
new_name = os.path.join(Config.TRAIN_DATASET_PATH, "BraTS20_Training_355/BraTS20_Training_355_seg.nii")
if os.path.exists(old_name): 
    os.rename(old_name, new_name)

# Get patient IDs and Split
train_and_val_directories = [f.path for f in os.scandir(Config.TRAIN_DATASET_PATH) if f.is_dir()]
train_and_test_ids = [d[d.rfind('/')+1:] for d in train_and_val_directories]

# Split data (70% train, 20% val, 10% test)
train_test_ids, val_ids = train_test_split(train_and_test_ids, test_size=0.2, random_state=42)
train_ids, test_ids = train_test_split(train_test_ids, test_size=0.125, random_state=42)

print(f"Splits -> Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

# Create Datasets
train_dataset = BraTSDataset(train_ids, Config.TRAIN_DATASET_PATH, 
                             img_size=Config.IMG_SIZE, volume_slices=Config.VOLUME_SLICES, volume_start=Config.VOLUME_START_AT)
val_dataset = BraTSDataset(val_ids, Config.TRAIN_DATASET_PATH, 
                           img_size=Config.IMG_SIZE, volume_slices=Config.VOLUME_SLICES, volume_start=Config.VOLUME_START_AT)
test_dataset = BraTSDataset(test_ids, Config.TRAIN_DATASET_PATH, 
                            img_size=Config.IMG_SIZE, volume_slices=Config.VOLUME_SLICES, volume_start=Config.VOLUME_START_AT)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")
print(f"✓ Test batches: {len(test_loader)}")

# Sanity check: Test loading one batch
sample_batch = next(iter(train_loader))
print(f"✓ Image batch shape: {sample_batch['image'].shape}")
print(f"✓ Mask batch shape: {sample_batch['mask'].shape}")

Splits -> Train: 258, Val: 74, Test: 37
✓ Valid patients: 258/258
✓ Valid patients: 74/74
✓ Valid patients: 37/37
✓ Train batches: 3225
✓ Val batches: 925
✓ Test batches: 463
✓ Image batch shape: torch.Size([8, 2, 128, 128])
✓ Mask batch shape: torch.Size([8, 128, 128])


In [11]:
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Model
model = BUNetMultiTask(
    encoder_name=Config.ENCODER_NAME,
    in_channels=Config.IN_CHANNELS,
    num_classes=Config.NUM_CLASSES,
    num_tumor_types=Config.NUM_TUMOR_TYPES
).to(Config.DEVICE)

# Loss and optimizer
loss_fn = smp.losses.DiceLoss(mode='multiclass')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


# Training loop
NUM_EPOCHS = 35

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    
    for batch_idx, batch in enumerate(train_loader):
        images = batch['image'].to(DEVICE)
        masks = batch['mask'].to(DEVICE)
        
        # Forward
        optimizer.zero_grad()
        
        # Unpack the tuple here
        seg_outputs, cls_outputs = model(images)
        
        # Calculate loss only on the segmentation outputs
        loss = loss_fn(seg_outputs, masks)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        # Print progress
        if batch_idx % 50 == 0:
            print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Batch [{batch_idx}/{len(train_loader)}] Loss: {loss.item():.4f}")
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(DEVICE)
            masks = batch['mask'].to(DEVICE)
            seg_outputs, cls_outputs = model(images)
            loss = loss_fn(seg_outputs, masks)
            val_loss += loss.item()
    
    # Epoch summary
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"{'='*60}\n")
    

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
        
        # Create the file path
        checkpoint_name = f'checkpoint_epoch_{epoch+1}.pth'
        save_path = os.path.join(Config.CHECKPOINT_DIR, checkpoint_name)
        
        # Save the file
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
        }, save_path)
        
        print(f"✓ Saved checkpoint: {save_path}\n")

Using device: mps
Epoch [1/35] Batch [0/3225] Loss: 0.8948
Epoch [1/35] Batch [50/3225] Loss: 0.6172
Epoch [1/35] Batch [100/3225] Loss: 0.6230
Epoch [1/35] Batch [150/3225] Loss: 0.6291
Epoch [1/35] Batch [200/3225] Loss: 0.3476
Epoch [1/35] Batch [250/3225] Loss: 0.3788
Epoch [1/35] Batch [300/3225] Loss: 0.3859
Epoch [1/35] Batch [350/3225] Loss: 0.3347
Epoch [1/35] Batch [400/3225] Loss: 0.3073
Epoch [1/35] Batch [450/3225] Loss: 0.6291
Epoch [1/35] Batch [500/3225] Loss: 0.3857
Epoch [1/35] Batch [550/3225] Loss: 0.3169
Epoch [1/35] Batch [600/3225] Loss: 0.5295
Epoch [1/35] Batch [650/3225] Loss: 0.3442
Epoch [1/35] Batch [700/3225] Loss: 0.1979
Epoch [1/35] Batch [750/3225] Loss: 0.1759
Epoch [1/35] Batch [800/3225] Loss: 0.2785
Epoch [1/35] Batch [850/3225] Loss: 0.4502
Epoch [1/35] Batch [900/3225] Loss: 0.3590
Epoch [1/35] Batch [950/3225] Loss: 0.2823
Epoch [1/35] Batch [1000/3225] Loss: 0.4328
Epoch [1/35] Batch [1050/3225] Loss: 0.2399
Epoch [1/35] Batch [1100/3225] Loss: 

KeyboardInterrupt: 

In [ ]:
# ── Evaluate & Print Results ─────────────────────────────────────────────────
print("Evaluating on test set...")
test_results = evaluate(model, test_loader, DEVICE)

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"  Dice  Background : {test_results['dice_class_0']:.4f}")
print(f"  Dice  NCR (1)    : {test_results['dice_class_1']:.4f}")
print(f"  Dice  Edema (2)  : {test_results['dice_class_2']:.4f}")
print(f"  Dice  ET  (3)    : {test_results['dice_class_3']:.4f}")
print(f"  Mean Tumor Dice  : {test_results['mean_tumor_dice']:.4f}  ← key metric")
print(f"  HD95  NCR (1)    : {test_results['hd95_class_1']:.2f} px")
print(f"  HD95  Edema (2)  : {test_results['hd95_class_2']:.2f} px")
print(f"  HD95  ET  (3)    : {test_results['hd95_class_3']:.2f} px")
print(f"  Mean Tumor HD95  : {test_results['mean_tumor_hd95']:.2f} px")
print("="*50)

# Ensure the results directory exists
os.makedirs(Config.RESULTS_DIR, exist_ok=True)

# Save the test results to a JSON file
results_path = os.path.join(Config.RESULTS_DIR, 'baseline_test_metrics.json')

with open(results_path, 'w') as f:
    json.dump(test_results, f, indent=4)

print(f"✓ Test metrics permanently saved to: {results_path}")

Evaluating on test set...

TEST SET RESULTS
  Dice  Background : 0.9952
  Dice  NCR (1)    : 0.6968
  Dice  Edema (2)  : 0.6669
  Dice  ET  (3)    : 0.7369
  Mean Tumor Dice  : 0.7002  ← key metric
  HD95  NCR (1)    : 1.97 px
  HD95  Edema (2)  : 6.00 px
  HD95  ET  (3)    : 5.23 px
  Mean Tumor HD95  : 4.18 px
